# Esquema: regresión lineal (MVP manual)

Versión **mínima viable**: CSV → pandas → **tratamiento manual** → split → **varios `Pipeline`** (mismo preprocesado manual, distinto modelo).

> En [07.b](../07.b-ejemplos-supervisados/) el imputer/encoder van **dentro** del `Pipeline`. Aquí el escalado+modelo sí van en pipeline; la limpieza va **a mano** en pandas.

| Paso | Qué hace |
|------|----------|
| 1–4 | CSV, target numérico, `fillna`, `get_dummies` |
| 5 | `train_test_split` |
| 6 | Comparar modelos: `StandardScaler` + regresores |

Siguiente: [07.b regresión](../07.b-ejemplos-supervisados/01-regresion-lineal.ipynb).


## 1. Importar CSV y revisar datos

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("data/datos_casas.csv")
print("Tipos:\n", df.dtypes)
print("\nFaltantes:\n", df.isna().sum())
display(df)


Tipos:
 Metros_Cuadrados    float64
Zona                 object
Precio                int64
dtype: object

Faltantes:
 Metros_Cuadrados    1
Zona                1
Precio              0
dtype: int64


,Metros_Cuadrados,Zona,Precio
0,45.0,Centro,125000
1,52.0,Periferia,138000
2,NaN,Centro,132000
3,61.0,Periferia,155000
4,70.0,Centro,172000
5,85.0,NaN,195000
6,95.0,Periferia,210000
7,110.0,Centro,245000
8,130.0,Periferia,285000
9,150.0,Centro,320000


## 2. Target numérico (`Precio`)

In [10]:
df = df.dropna(subset=["Precio"]).copy()
y = df["Precio"]
print("Filas:", len(df))


Filas: 11


## 3–4. Features (manual)

In [11]:
metros = df["Metros_Cuadrados"].astype(float)
X_num = pd.DataFrame({"Metros_Cuadrados": metros.fillna(metros.median())})

zona = df["Zona"].fillna("Desconocida").astype(str)
X_cat = pd.get_dummies(zona, prefix="Zona", dtype=float)

X = pd.concat([X_num, X_cat], axis=1)
display(X.head())


,Metros_Cuadrados,Zona_Centro,Zona_Desconocida,Zona_Periferia
0,45.0,1.0,0.0,0.0
1,52.0,0.0,0.0,1.0
2,90.0,1.0,0.0,0.0
3,61.0,0.0,0.0,1.0
4,70.0,1.0,0.0,0.0


## 5. Split train / test

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 6. Varios modelos en Pipeline

Mismo `StandardScaler` para todos; solo cambia el estimador final. Bucle mínimo para comparar **R²** y **MSE** en test.


In [13]:
def build_models():
    """Misma lista que 07.b (comenta entradas para excluir modelos)."""
    from sklearn.ensemble import (
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )
    from sklearn.linear_model import Lasso, LinearRegression, Ridge
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.tree import DecisionTreeRegressor
    from xgboost import XGBRegressor
    from catboost import CatBoostRegressor

    return {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(random_state=RANDOM_STATE),
        "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=5000),
        "DecisionTree": DecisionTreeRegressor(
            criterion="squared_error",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=100,
            criterion="squared_error",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=1.0,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
        "KNN": KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
        "XGBoost": XGBRegressor(
            random_state=RANDOM_STATE, verbosity=0, n_estimators=100, n_jobs=-1
        ),
        "CatBoost": CatBoostRegressor(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
        ),
    }


RANDOM_STATE = 42
MODELS = build_models()


filas = []
for nombre, modelo in MODELS.items():
    pipe = make_pipeline(StandardScaler(), modelo)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    filas.append(
        {
            "modelo": nombre,
            "R2": r2_score(y_test, pred),
            "MSE": mean_squared_error(y_test, pred),
        }
    )

tabla = pd.DataFrame(filas).sort_values("R2", ascending=False)
display(tabla.round(4))

mejor = tabla.iloc[0]["modelo"]
print(f"\nMejor R² en test: {mejor}")


,modelo,R2,MSE
2,Lasso,0.9334,4.331365e+08
0,LinearRegression,0.9298,4.564532e+08
1,Ridge,0.9204,5.179841e+08
5,RandomForest,0.8377,1.055902e+09
4,DecisionTree,0.7252,1.787667e+09
3,KNN,-0.1628,7.564370e+09



Mejor R² en test: Lasso
